# 04 - Optimization: EMS Unit Allocation Policies

This notebook implements and compares three allocation policies:
- **P0 (Spatial Baseline)**: Spatially-stratified uniform allocation
- **P1 (Demand-Proportional)**: Units allocated proportional to demand
- **P2 (Optimized)**: Demand-weighted MIP solved with PuLP/CBC

Solves for multiple fleet sizes K = {15, 20, 25, 30, 35, 40} with capacity constraints.

**Runtime:** ~5 minutes  
**Data Required:** Distance matrix, demand rates, firehouses

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally")
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

## Load Data for Optimization

In [ ]:
import yaml

# Load distance and demand data
dm = pd.read_csv(os.path.join(PROCESSED_DIR, 'distance_matrix_firehouse_precinct.csv'), index_col=0)
dm.columns = dm.columns.astype(str)
firehouses = pd.read_csv(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv'))
precinct_demand = pd.read_csv(os.path.join(PROCESSED_DIR, 'demand_lambda_precinct.csv'))

# Load optimization config
with open(os.path.join(CONFIGS_DIR, 'optimization.yaml')) as f:
    opt_config = yaml.safe_load(f)

print(f"Distance matrix: {dm.shape}")
print(f"Firehouses: {len(firehouses)}")
print(f"Precincts: {len(precinct_demand)}")
print(f"Capacity constraint: {opt_config.get('firehouse_capacity', 2)}")
print(f"Coverage threshold: {opt_config.get('coverage_threshold_minutes', 8)} min")

# Build travel time matrix (no ToD for optimization)
from ems_readiness.service.travel_time import build_travel_time_matrix
tt = build_travel_time_matrix(dm, speed_mph=20.0, hour_of_day=None)
print(f"\nTravel time matrix: mean={tt.values.mean():.2f} min, max={tt.values.max():.2f} min")

# Demand vector
demand = precinct_demand.set_index('precinct')['lambda_per_hour']
demand.index = demand.index.astype(str)
print(f"\nDemand vector: {len(demand)} precincts, total={demand.sum():.4f} crashes/hour")

## Policy Implementations
### P0: Spatial Baseline (Uniform)

In [ ]:
from ems_readiness.optimization.policies import uniform_allocation, demand_proportional_allocation

K_values = [15, 20, 25, 30, 35, 40]
capacity = opt_config.get('firehouse_capacity', 2)
all_results = {}

for K in K_values:
    p0 = uniform_allocation(dm.index.tolist(), K=K, capacity=capacity)
    all_results[f'P0_K{K}'] = p0
    n_active = (p0 > 0).sum()
    print(f"P0 K={K}: {n_active} active firehouses, total units={p0.sum()}")

### P1: Demand-Proportional

In [ ]:
for K in K_values:
    p1 = demand_proportional_allocation(tt, demand, K=K, capacity=capacity)
    all_results[f'P1_K{K}'] = p1
    n_active = (p1 > 0).sum()
    print(f"P1 K={K}: {n_active} active firehouses, total units={p1.sum()}")

### P2: Demand-Weighted Optimization (MIP)

In [ ]:
from ems_readiness.optimization.models import build_demand_weighted, extract_allocation
import pulp

for K in K_values:
    print(f"\nSolving P2 for K={K}...")
    prob = build_demand_weighted(tt, demand, K=K, capacity=capacity)
    solver = pulp.PULP_CBC_CMD(msg=0, timeLimit=120)
    prob.solve(solver)

    status = pulp.LpStatus[prob.status]
    obj = pulp.value(prob.objective)
    print(f"  Status: {status}, Objective: {obj:.6f}")

    alloc = extract_allocation(prob)
    all_results[f'P2_K{K}'] = alloc
    n_active = (alloc > 0).sum()
    print(f"  Active firehouses: {n_active}, Total units: {alloc.sum()}")

## Compare Policies

In [ ]:
comparison_rows = []
for K in K_values:
    for policy in ['P0', 'P1', 'P2']:
        key = f'{policy}_K{K}'
        alloc = all_results[key]
        # Compute expected response time
        # For each precinct, find nearest firehouse with units
        active_fh = alloc[alloc > 0].index
        if len(active_fh) > 0:
            tt_active = tt.loc[active_fh]
            min_tt = tt_active.min(axis=0)  # min travel time per precinct
            # Demand-weighted average
            common_prec = [p for p in demand.index if p in min_tt.index]
            if common_prec:
                d = demand[common_prec]
                t = min_tt[common_prec]
                weighted_rt = (d * t).sum() / d.sum()
            else:
                weighted_rt = float('inf')
        else:
            weighted_rt = float('inf')

        comparison_rows.append({
            'Policy': policy,
            'K': K,
            'Active Firehouses': int((alloc > 0).sum()),
            'Weighted Avg Travel Time (min)': round(weighted_rt, 3),
            'Max Allocation': int(alloc.max()),
        })

comp_df = pd.DataFrame(comparison_rows)
display(comp_df)

### Visualization: Policy Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for policy in ['P0', 'P1', 'P2']:
    subset = comp_df[comp_df['Policy'] == policy]
    axes[0].plot(subset['K'], subset['Weighted Avg Travel Time (min)'], '-o', label=policy, markersize=6)

axes[0].set_xlabel('Fleet Size (K)')
axes[0].set_ylabel('Demand-Weighted Avg Travel Time (min)')
axes[0].set_title('Policy Comparison: Travel Time vs Fleet Size')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for policy in ['P0', 'P1', 'P2']:
    subset = comp_df[comp_df['Policy'] == policy]
    axes[1].plot(subset['K'], subset['Active Firehouses'], '-s', label=policy, markersize=6)

axes[1].set_xlabel('Fleet Size (K)')
axes[1].set_ylabel('Active Firehouses')
axes[1].set_title('Active Firehouses vs Fleet Size')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
save_output(fig, 'policy_comparison.png', 'figures/optimization')
plt.show()

### Allocation Heatmaps

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
K_show = 20

for idx, policy in enumerate(['P0', 'P1', 'P2']):
    key = f'{policy}_K{K_show}'
    alloc = all_results[key]
    active = alloc[alloc > 0].sort_values(ascending=True)

    axes[0, idx].barh(range(len(active)), active.values, color=['steelblue', 'darkorange', 'seagreen'][idx])
    axes[0, idx].set_yticks(range(len(active)))
    axes[0, idx].set_yticklabels(active.index, fontsize=6)
    axes[0, idx].set_xlabel('Units')
    axes[0, idx].set_title(f'{policy} (K={K_show}): {len(active)} firehouses')

K_show2 = 30
for idx, policy in enumerate(['P0', 'P1', 'P2']):
    key = f'{policy}_K{K_show2}'
    alloc = all_results[key]
    active = alloc[alloc > 0].sort_values(ascending=True)

    axes[1, idx].barh(range(len(active)), active.values, color=['steelblue', 'darkorange', 'seagreen'][idx])
    axes[1, idx].set_yticks(range(len(active)))
    axes[1, idx].set_yticklabels(active.index, fontsize=6)
    axes[1, idx].set_xlabel('Units')
    axes[1, idx].set_title(f'{policy} (K={K_show2}): {len(active)} firehouses')

plt.tight_layout()
save_output(fig, 'allocation_heatmaps.png', 'figures/optimization')
plt.show()

## Save Allocations for Simulation

In [ ]:
# Save allocations for use by simulation notebook
alloc_dir = os.path.join(RESULTS_DIR, 'optimization')
os.makedirs(alloc_dir, exist_ok=True)

for K in K_values:
    alloc_df = pd.DataFrame({
        'P0': all_results[f'P0_K{K}'],
        'P1': all_results[f'P1_K{K}'],
        'P2': all_results[f'P2_K{K}'],
    })
    path = os.path.join(alloc_dir, f'allocations_K{K}.csv')
    alloc_df.to_csv(path)
    print(f"Saved: {path}")

print("\nAllocations saved for simulation phase.")

## Summary

- P2 (demand-weighted optimization) consistently achieves lowest weighted travel time
- P1 (demand-proportional) is a strong heuristic, close to P2
- P0 (uniform baseline) has highest travel times due to ignoring demand
- Capacity constraint of 2 units/firehouse forces spatial dispersion
- Diminishing returns for K > 30 (already approaching full firehouse utilization)